In [1]:
%load_ext autoreload
%autoreload 2

# assignment

Week 10 강의자료 Section 3의 BERT for Token Classification 파트가 구현된 ipynb 파일 제출

### 최신 버전의 transformers에서 아래와 같은 문제가 발생함

- 이에 평소에 사용하던 버전으로 설치해 해결함

[관련 링크](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct/discussions/126)

In [2]:
!pip uninstall torch datasets transformers accelerate -y
!pip install torch==2.8.0 --index-url=https://download.pytorch.org/whl/cu128
!pip install datasets transformers==4.46.3 evaluate seqeval accelerate==0.26.0

Found existing installation: torch 2.8.0+cu128
Uninstalling torch-2.8.0+cu128:
  Successfully uninstalled torch-2.8.0+cu128
Found existing installation: datasets 4.4.1
Uninstalling datasets-4.4.1:
  Successfully uninstalled datasets-4.4.1
Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Successfully uninstalled transformers-4.46.3
Found existing installation: accelerate 0.26.0
Uninstalling accelerate-0.26.0:
  Successfully uninstalled accelerate-0.26.0
Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached https://download.pytorch.org/whl/cu128/torch-2.8.0%2Bcu128-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (30 kB)
Using cached https://download.pytorch.org/whl/cu128/torch-2.8.0%2Bcu128-cp311-cp311-manylinux_2_28_x86_64.whl (889.2 MB)
  Using cached datasets-4.4.1-py3-none-any.whl.metadata (19 kB)
  Using cached transformers-4.46.3-py3-none-any.whl.metadata (44 kB)
  Using cached accelerate-0.26.0-py3-none-any.whl.metadata (18 kB

In [3]:
from datasets import load_dataset

ner_data = load_dataset("lhoestq/conll2003")

print("[conll2003 data]: ", ner_data)
print("[conll2003 first sentence's tokens]: ",ner_data["train"][0]["tokens"])
print("[conll2003 first sentence's labels]: ",ner_data["train"][0]["ner_tags"])

label_names = ['O','B-PER','I-PER','B-ORG','I-ORG','B-LOC','I-LOC','B-MISC', 'I-MISC']

print("[conll2003 labels info]: ",label_names)

[conll2003 data]:  DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})
[conll2003 first sentence's tokens]:  ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
[conll2003 first sentence's labels]:  [3, 0, 7, 0, 0, 0, 7, 0, 0]
[conll2003 labels info]:  ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [4]:
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [5]:
print("[conll2003 100th sentence's tokens]: ", ner_data["train"][99]["tokens"])
print("[conll2003 100th sentence's labels]: ", ner_data["train"][99]["ner_tags"])
print("[conll2003 100th sentence's length]: ", len(ner_data["train"][99]["ner_tags"]))

encoded = tokenizer(
    ner_data["train"][99]["tokens"], truncation=True, is_split_into_words=True
)
print("[Tokenized 100th sentence]: ", encoded)
print("[Tokenized 100th sentence's length]: ", len(encoded.input_ids))
print("[Decoded 100th sentence]: ", tokenizer.convert_ids_to_tokens(encoded.input_ids))

[conll2003 100th sentence's tokens]:  ['The', 'Syrians', 'are', 'confused', ',', 'they', 'are', 'definitely', 'tense', ',', 'but', 'the', 'general', 'assessment', 'here', 'in', 'Washington', 'is', 'that', 'this', 'is', 'essentially', 'a', 'storm', 'in', 'a', 'teacup', ',', '"', 'he', 'said', '.']
[conll2003 100th sentence's labels]:  [0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[conll2003 100th sentence's length]:  32
[Tokenized 100th sentence]:  {'input_ids': [101, 1109, 8697, 1116, 1132, 4853, 117, 1152, 1132, 5397, 8901, 117, 1133, 1103, 1704, 8670, 1303, 1107, 1994, 1110, 1115, 1142, 1110, 7588, 170, 4162, 1107, 170, 5679, 18637, 117, 107, 1119, 1163, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
[Tokenized 100th 

In [6]:
def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)
        elif word_id is None:
            new_labels.append(-100)
        else:
            label = labels[word_id]
            if label % 2 == 1:
                label += 1
            new_labels.append(label)
    return new_labels

In [7]:
def tokenize(examples):
    tokenized_inputs = tokenizer(
    examples["tokens"], truncation=True, is_split_into_words=True
    )
    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))
    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

In [8]:
ner_encoded = ner_data.map(tokenize, batched=True, remove_columns=ner_data["train"].column_names)

In [9]:
ner_encoded

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

In [10]:
print("[Tokenized 100th sentence]: ", ner_encoded["train"][99]["input_ids"])
print("[Decoded 100th sentence]: ", tokenizer.convert_ids_to_tokens(ner_encoded["train"][99]["input_ids"]))
print("[100th sentence's new_label]: ", ner_encoded["train"][99]["labels"])

[Tokenized 100th sentence]:  [101, 1109, 8697, 1116, 1132, 4853, 117, 1152, 1132, 5397, 8901, 117, 1133, 1103, 1704, 8670, 1303, 1107, 1994, 1110, 1115, 1142, 1110, 7588, 170, 4162, 1107, 170, 5679, 18637, 117, 107, 1119, 1163, 119, 102]
[Decoded 100th sentence]:  ['[CLS]', 'The', 'Syrian', '##s', 'are', 'confused', ',', 'they', 'are', 'definitely', 'tense', ',', 'but', 'the', 'general', 'assessment', 'here', 'in', 'Washington', 'is', 'that', 'this', 'is', 'essentially', 'a', 'storm', 'in', 'a', 'tea', '##cup', ',', '"', 'he', 'said', '.', '[SEP]']
[100th sentence's new_label]:  [-100, 0, 7, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100]


In [11]:
from transformers import AutoModelForTokenClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForTokenClassification.from_pretrained(model_name, id2label=id2label, label2id=label2id).to(device)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
import numpy as np
import evaluate

metric = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./NER_trained_models",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="none",
)

In [14]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ner_encoded["train"],
    eval_dataset=ner_encoded["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)
trainer.train()

/tmp/ipykernel_1923448/2873445360.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/home/jpong/miniconda3/envs/applied_nlp/lib/python3.11/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.066016,0.883713,0.925951,0.904339,0.981515
2,0.189700,0.060792,0.911572,0.938573,0.924876,0.984223
3,0.048200,0.055949,0.923799,0.944631,0.934099,0.985224


/home/jpong/miniconda3/envs/applied_nlp/lib/python3.11/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/jpong/miniconda3/envs/applied_nlp/lib/python3.11/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/jpong/miniconda3/envs/applied_nlp/lib/python3.11/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=1317, training_loss=0.09799907962390665, metrics={'train_runtime': 136.8386, 'train_samples_per_second': 307.83, 'train_steps_per_second': 9.624, 'total_flos': 1163294308216332.0, 'train_loss': 0.09799907962390665, 'epoch': 3.0})

In [16]:
preds_output = trainer.predict(ner_encoded["validation"])
preds_output.metrics

/home/jpong/miniconda3/envs/applied_nlp/lib/python3.11/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'test_loss': 0.05594947189092636,
 'test_precision': 0.923798551678736,
 'test_recall': 0.9446314372265231,
 'test_f1': 0.9340988517224164,
 'test_accuracy': 0.9852239948195679,
 'test_runtime': 5.7051,
 'test_samples_per_second': 569.67,
 'test_steps_per_second': 17.879}